# Lesson 9: The demo page, from your Drive to a public website

Today you publish FieldSense as a **website judges can open**, with a link like
`https://setarehalamdar.github.io/FieldSense/`.

**How it works (remember the restaurant analogy):** there's no waiter (API) and no kitchen running on a server. The page is three plain files (`index.html`, `style.css`, `app.js`) plus small **data files**, all hosted for free by **GitHub Pages**. When someone opens the page, their browser downloads the files and draws everything itself: the barcode, the odds squares, the trade-off table, the explanations, the evidence passport.

It works because of the **data contract** you built in Lessons 5–7: the page only needs the field profiles.

**Steps today**
1. Install the latest code and run the tests.
2. **Export** a *public* copy of your profiles. This is the privacy gate.
3. **Preview** the page inside Colab with your real data.
4. **Publish**: upload the data files to GitHub with a pull request.
5. **Switch on GitHub Pages** and check the live page.

**Job skills:** static sites vs servers, privacy by design, publishing through pull requests, and checking a page like a user would (phone, dark mode, every field).

**Time:** about 45 minutes.


## Step 0: Review and merge pull request #4

Open https://github.com/SetarehAlamdar/FieldSense/pulls, find the **Lesson 9** pull request, and review it with the five-pass method:
- **`src/fieldsense/export.py`**: the privacy gate. What does it remove or round, and what does `check_public` refuse?
- **`tests/test_export.py`**: which test proves that coordinates get rounded? Which one proves that the original profile isn't changed?
- **`docs/`**: the website. You don't need to understand every line of `app.js`. Read the comment at the top and the function names (`renderBarcode`, `renderOdds`, ...): each one draws one section of the page.

Then **merge**.


## Project setup: run this first, every session

Everything we save, **data files and code modules**, goes into your **Google Drive** under `MyDrive/fieldsense/`. Colab wipes its own disk whenever a session ends, but Drive keeps everything, so each lesson can pick up the previous lessons' files.

Colab will ask permission to access your Drive the first time each session; click **Allow**.


In [ ]:
# ---- Project setup: run this first, every session ----
import os, sys, glob, shutil
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/fieldsense"
except ImportError:                       # running outside Colab: use a local folder instead
    PROJECT_DIR = os.path.abspath("fieldsense")
CACHE_DIR = os.path.join(PROJECT_DIR, "data_cache")
os.makedirs(CACHE_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)                     # %%writefile now saves code modules into Drive too
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)       # lets `import fieldsense_data` find them

# One-time rescue: copy anything still sitting in Colab's temporary folder into Drive
for f in glob.glob("/content/data_cache/*"):
    if not os.path.exists(os.path.join(CACHE_DIR, os.path.basename(f))):
        shutil.copy(f, CACHE_DIR)
        print("Rescued into Drive:", os.path.basename(f))

print("Project folder:", PROJECT_DIR)
print("Saved data files:", sorted(os.listdir(CACHE_DIR)) or "none yet")

PRIVATE_DIR = os.path.join(PROJECT_DIR, "private")       # lab records: NEVER goes to GitHub
PROFILE_DIR = os.path.join(PROJECT_DIR, "profiles")      # field profiles + evidence registry
DEMO_DIR = os.path.join(PROJECT_DIR, "demo")             # figures for slides and website
for d in (PRIVATE_DIR, PROFILE_DIR, DEMO_DIR):
    os.makedirs(d, exist_ok=True)

## Step 1: Install and test

You should see **49 passed**.


In [ ]:
import subprocess
REPO_URL = "https://github.com/SetarehAlamdar/FieldSense.git"
REPO_DIR = os.environ.get("FIELDSENSE_REPO", "/content/FieldSense")

def sh(cmd):
    """Run a shell command; print its output; stop the notebook if it fails."""
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print((r.stdout + r.stderr).strip()[-1500:])
    if r.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}")

url = REPO_URL
try:                                           # private repo: use a token from Colab Secrets
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{token}@")
except Exception:
    pass                                       # public repo (or not in Colab): no token needed

if os.path.exists(os.path.join(REPO_DIR, ".git")):
    sh(f"git -C {REPO_DIR} pull --ff-only")
else:
    sh(f"git clone --quiet {url} {REPO_DIR}")
sh(f'pip install -q -e "{REPO_DIR}[dev]"')

SRC = os.path.join(REPO_DIR, "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)
import fieldsense as fs
print("fieldsense version", fs.__version__, "from", os.path.dirname(fs.__file__))
sh(f"git -C {REPO_DIR} log --oneline -5")
sh(f"cd {REPO_DIR} && python -m pytest -q")

## Step 2: Export the public data (the privacy gate)

Your profiles in Drive are **private working files**: they contain the plots' exact survey-map centres and areas. A website is public forever (people save copies), so we publish a *cleaned copy* instead:

| In your Drive profile | On the public page |
|---|---|
| Exact plot centre (5 decimals, about 1 m) | Rounded to 2 decimals (about 1 km): "near Elora" |
| Plot area from the survey map | Removed |
| `location_source: survey map` | Removed |
| Barcode: only a picture file name | The greenness numbers, so the page can draw it |

Then `check_public` scans every file again and **refuses to write anything that still looks private**: an unrounded coordinate, a private field, or text mentioning a `PRIVATE` file. This is called **privacy by design**: the safe behaviour is built into the code, so it doesn't depend on someone remembering to be careful at 2 a.m. during a hackathon.

What stays: satellite-derived crop history, climate odds, soil, rotation options and **aggregate** validation scores ("wheat found in 2 of 2"). None of these are the lab's records. If you want to be extra careful, show the page to the lab lead before publishing.


In [ ]:
import json, shutil
from fieldsense.export import export_demo, check_public

SITE_DIR = os.path.join(PROJECT_DIR, "site_data")
shutil.rmtree(SITE_DIR, ignore_errors=True)
written = export_demo(PROFILE_DIR, os.path.join(CACHE_DIR, "hls_plot_ndvi.csv"), SITE_DIR)
for f in written:
    print(f"{os.path.basename(f):15s} {os.path.getsize(f)/1024:6.1f} KB")

sample = json.load(open(os.path.join(SITE_DIR, "plot_3.json")))
print("\nPublic field info:", sample["field"])
print("Privacy check on all files:", "OK" if not any(check_public(json.load(open(f))) for f in written) else "PROBLEM")

**Check:** the field info shows `lat` and `lon` with only 2 decimals, and no `area_ha`. Each plot file is around 30 KB, small enough to load instantly even on a phone.


## Step 3: Preview the page inside Colab

Before publishing anything, look at it. This cell:
1. copies the website files from the repository and your exported data into one folder,
2. starts a tiny **web server** in the background (`python -m http.server`). Browsers only load data files from a web address, not from a folder, which is why opening `index.html` straight from your computer shows "No field data found",
3. asks Colab to show that server in a window.

Click through all four plots, hover over the barcode, and scroll to the end.


In [ ]:
import subprocess, time
PREVIEW = "/content/site_preview"
shutil.rmtree(PREVIEW, ignore_errors=True)
shutil.copytree(os.path.join(REPO_DIR, "docs"), PREVIEW)
shutil.copytree(SITE_DIR, os.path.join(PREVIEW, "data"), dirs_exist_ok=True)
server = subprocess.Popen(["python", "-m", "http.server", "8000", "--directory", PREVIEW],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1)
try:
    from google.colab import output
    output.serve_kernel_port_as_window(8000, path="/index.html")
    print("Click the link above to open the preview in a new tab.")
except ImportError:
    print("Not in Colab: open http://localhost:8000 in your browser.")

**Review it as a judge would.** For each one, write yes or no:
- [ ] Is it clear in **10 seconds** what FieldSense does?
- [ ] Do plots 1–2 (simple) and 3–4 (diverse) look clearly different in the barcode?
- [ ] Is "In 2026" right for each plot? (Plots 3–4 should say winter wheat.)
- [ ] Does every number have an evidence badge, and does it make sense?
- [ ] Is anything confusing or too technical for a farmer?

Note what you'd change. Text changes are easy: they're all in `docs/index.html` (headings, introductions) or `docs/app.js` (labels like "Rotation now").


## Step 4: Publish the data through a pull request

The website files are already in the repository (you merged them in Step 0). Now the **data files** need to go into `docs/data/`. We'll do it **through a pull request, using only the GitHub website**, the same review habit as for code, because data is published too.

1. Run the next cell. It packs the 6 data files into `fieldsense_site_data.zip` and downloads it to your computer. **Unzip it.**
2. On GitHub, open the repository → the **`docs`** folder → the **`data`** folder.
3. Click **Add file → Upload files**, and drag in the 6 unzipped files (`index.json`, `evidence.json`, `plot_1.json` … `plot_4.json`).
4. At the bottom, choose **"Create a new branch for this commit and start a pull request"**, name it `demo-data`, and click **Propose changes**, then **Create pull request**.
5. Look at **Files changed** one last time (a final privacy check), then **Merge**.


In [ ]:
zip_path = shutil.make_archive("/content/fieldsense_site_data", "zip", SITE_DIR)
print("Created", zip_path, f"({os.path.getsize(zip_path)/1024:.0f} KB)")
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    pass

## Step 5: Switch on GitHub Pages (once)

1. On GitHub: repository → **Settings** → **Pages** (left menu).
2. Under **Build and deployment → Source**, choose **Deploy from a branch**.
3. **Branch:** `main`, **folder:** `/docs` → **Save**.
4. Wait 1–2 minutes and refresh. GitHub shows *"Your site is live at https://setarehalamdar.github.io/FieldSense/"*.

From now on, **every merge to `main` updates the live site automatically**, usually within a minute. That's a simple form of **continuous deployment**.

**Final checks on the live link:**
- [ ] Open it **on your phone**. Is everything readable, and does nothing need sideways scrolling?
- [ ] Switch your phone or computer to **dark mode**. The page follows it.
- [ ] Click **every plot**.
- [ ] Send the link to one person who doesn't know the project, and ask what they understood. That's the best 5-minute test you can do.


## What you learned

- **Static site vs server:** a page that reads files can't crash or run up a bill, which makes it ideal for a demo. An API is needed when data is too big to ship to the browser, or must be computed on request.
- **Privacy by design:** an export step that rounds, removes, and *refuses* unsafe output, plus tests that prove it.
- **Publishing data through pull requests**, so data gets the same review as code.
- **Continuous deployment:** merge to `main`, and the site updates by itself.
- **Checking like a user:** phone, dark mode, every option, and a fresh pair of eyes.

## Interview practice

**Q1. "When would you serve results from a static site instead of building an API?"**
<details><summary>Model answer</summary>

When the data is small, changes rarely, and is the same for every user: precompute it, publish files (often through a CDN), and the browser renders them. It's cheap, fast and very reliable. Build an API when data is large or per-user, needs authentication, must be computed on demand (e.g. a new field's coordinates), or changes often. A common pattern is both: precomputed files for popular views, an API for everything else.
</details>

**Q2. "How do you make sure no sensitive data ends up in a public dashboard?"**
<details><summary>Model answer</summary>

Make it structural, not a matter of care: a single export step that whitelists or removes fields, coarsens what must stay (rounding coordinates, aggregating small counts), and validates the output, refusing to publish if a check fails. Test that step. Review published data like code (through pull requests). And minimise: publish only what the page actually needs.
</details>
